In [1]:
!pip install pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 21.0 MB/s eta 0:00:00


In [2]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

In [3]:
from pinecone import Pinecone

api_key = os.environ["PINECONE_API_KEY"]


pc = Pinecone(api_key=api_key)


In [4]:
query = "Tell me about Apple's products"
documents = [
    "Apple is a popular fruit that is often red or green.",
    "Apple Inc. released the iPhone 15 in 2023.",
    "An apple a day keeps the doctor away.",
    "Apple's M1 chip revolutionized the laptop industry.",
    "Baking an apple pie requires fresh apples and cinnamon."
]


In [5]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
   model="bge-reranker-v2-m3",
   query=query,
   documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
   top_n=3  # e.g., 3
)

In [7]:
print(reranked)


RerankResult(
  model='bge-reranker-v2-m3',
  data=[{
    index=0,
    score=0.04138472,
    document={
        id='0',
        text='Apple is a popular fruit that is often red or green.'
    }
  },{
    index=1,
    score=0.032713126,
    document={
        id='1',
        text='Apple Inc. released the iPhone 15 in 2023.'
    }
  },{
    index=3,
    score=0.019681409,
    document={
        id='3',
        text="Apple's M1 chip revolutionized the laptop industry."
    }
  }],
  usage={'rerank_units': 1}
)


In [8]:
def show_reranked(query, data):
    print(f"Query: {query}")
    for i, m in enumerate(data):
        print(f"{i+1}. Score: {m['score']:.3f} | Text: {m['document']['text']}")

show_reranked(query, reranked.data)


Query: Tell me about Apple's products
1. Score: 0.041 | Text: Apple is a popular fruit that is often red or green.
2. Score: 0.033 | Text: Apple Inc. released the iPhone 15 in 2023.
3. Score: 0.020 | Text: Apple's M1 chip revolutionized the laptop industry.


In [9]:
!pip install pandas torch transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [17]:
import os, time
import pandas as pd
import torch
from pinecone import Pinecone, ServerlessSpec

cloud = "aws"
region = "us-east-1"
spec = ServerlessSpec(cloud=cloud, region=region)  # CORRECT pour v6.x

index_name = "pinecone-reranker"
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])



In [18]:
if pc.has_index(index_name):
    pc.delete_index(index_name)


dimension = 384

pc.create_index(
    name=index_name,
    dimension=dimension,
    spec=spec
)

{
    "name": "pinecone-reranker",
    "metric": "cosine",
    "host": "pinecone-reranker-n70u3xb.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [19]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    file_path = os.path.join(tmpdir, "sample_notes_data.jsonl")
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    resp = requests.get(url)
    resp.raise_for_status()
    open(file_path, "wb").write(resp.content)
    df = pd.read_json(file_path, orient='records', lines=True)

print(df.head())


     id                                             values  \
0  P011  [-0.2027486265, 0.2769146562, -0.1509393603, 0...   
1  P001  [0.1842793673, 0.4459365904, -0.0770567134, 0....   
2  P002  [-0.2040648609, -0.1739618927, -0.2897160649, ...   
3  P003  [0.1889383644, 0.2924542725, -0.2335938066, -0...   
4  P004  [-0.12171068040000001, 0.1674752235, -0.231888...   

                                            metadata  
0  {'advice': 'rest, hydrate', 'symptoms': 'heada...  
1  {'tests': 'EKG, stress test', 'symptoms': 'che...  
2  {'HbA1c': '7.2', 'condition': 'diabetes', 'med...  
3  {'symptoms': 'cough, wheezing', 'diagnosis': '...  
4  {'referral': 'dermatology', 'condition': 'susp...  


In [20]:
index = pc.Index(index_name)
index.upsert_from_dataframe(df)

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

In [21]:
def is_ready(idx):
   stats = idx.describe_index_stats()
   return stats.total_vector_count > 0

while not is_ready(index):
   time.sleep(5)
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}


In [22]:
from sentence_transformers import SentenceTransformer

# Charger le modèle une seule fois
st_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    return st_model.encode(text)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [23]:
question = "what if my patient has leg pain"
emb = get_embedding(question)
print(emb.shape)  # Doit afficher (384,)


(384,)


In [25]:
import numpy as np

# On convertit explicitement le numpy array en liste Python natif
results = pc.Index(index_name).query(
    vector=emb.tolist(),
    top_k=5,
    include_metadata=True
)


In [26]:
question = "what if my patient has leg pain"
emb = get_embedding(question)

results = pc.Index(index_name).query(
    vector=emb.tolist(),
    top_k=5,                # nombre de résultats voulus
    include_metadata=True    # pour afficher les infos médicales
)

# Trier du plus pertinent au moins pertinent
matches = sorted(results.matches, key=lambda m: m.score, reverse=True)


In [27]:
def show_results(q, matches):
    print(f"Question: {q}")
    for i, m in enumerate(matches):
        print(f"{i+1}. ID: {m.id} | Score: {m.score:.3f} | Metadata: {m.metadata}")

show_results(question, matches)


Question: what if my patient has leg pain
1. ID: P0100 | Score: 0.518 | Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}
2. ID: P047 | Score: 0.501 | Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}
3. ID: P095 | Score: 0.501 | Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}
4. ID: P007 | Score: 0.460 | Metadata: {'surgery': 'knee arthroscopy', 'symptoms': 'pain, swelling', 'treatment': 'physical therapy'}
5. ID: P028 | Score: 0.447 | Metadata: {'condition': 'knee pain', 'referral': 'orthopedics'}


In [28]:
rerank_docs = [
    {
        "id": m.id,
        "reranking_field": "; ".join([f"{k}: {v}" for k, v in m.metadata.items()])
    }
    for m in matches
]


In [29]:
rerank_query = "which notes mention a vascular procedure?"


In [30]:
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=rerank_query,
    documents=rerank_docs,
    rank_fields=["reranking_field"],
    top_n=3  # nombre de notes rerankées à afficher
)


In [31]:
def show_reranked(q, matches):
    print(f"Refined Query: {q}")
    for i, m in enumerate(reranked.data):
        print(f"{i+1}. ID: {m['document']['id']} | Score: {m['score']:.3f} | Field: {m['document']['reranking_field']}")

show_reranked(rerank_query, reranked.data)


Refined Query: which notes mention a vascular procedure?
1. ID: P007 | Score: 0.001 | Field: surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical therapy
2. ID: P028 | Score: 0.000 | Field: condition: knee pain; referral: orthopedics
3. ID: P0100 | Score: 0.000 | Field: advice: over-the-counter pain relief, stretching; symptoms: muscle pain
